In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Display formatting for numpy
np.set_printoptions(precision=4, suppress=True)

# ---------------------- DATA PREPARATION ---------------------
features = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

targets = np.array([
    [0],
    [0],
    [0],
    [1]
], dtype=float)

print("Feature Shape:", features.shape)
print("Target Shape:", targets.shape)

# ---------------------- DATA VISUALIZATION ----------------------
plt.figure(figsize=(5, 4))
for idx, point in enumerate(features):
    plt.scatter(point[0], point[1], s=180)
    plt.text(point[0] + 0.02, point[1] + 0.02,
             f"label={int(targets[idx, 0])}")

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Logical AND Dataset")
plt.grid(True)
plt.show()

# ---------------------- ACTIVATION FUNCTIONS ----------------------
def relu_fn(x):
    return np.maximum(0, x)


def relu_grad(x):
    return (x > 0).astype(float)


def sigmoid_fn(x):
    return 1.0 / (1.0 + np.exp(-x))

# ---------------------- LOSS FUNCTION ----------------------
def bce_loss(true_val, pred_val, eps=1e-8):
    pred_val = np.clip(pred_val, eps, 1 - eps)
    return -np.mean(true_val * np.log(pred_val) +
                    (1 - true_val) * np.log(1 - pred_val))

# ---------------------- FORWARD PROPAGATION ----------------------
def forward_step(inp, w1, b1, w2, b2):
    hidden_linear = inp @ w1 + b1
    hidden_output = relu_fn(hidden_linear)

    final_linear = hidden_output @ w2 + b2
    predictions = sigmoid_fn(final_linear)

    memory = {
        "hidden_linear": hidden_linear,
        "hidden_output": hidden_output,
        "predictions": predictions
    }
    return predictions, memory

# ---------------------- BACKWARD PROPAGATION ----------------------
def backward_step(inp, true_val, memory, w2):
    h_lin = memory["hidden_linear"]
    h_out = memory["hidden_output"]
    pred = memory["predictions"]

    error_output = pred - true_val

    grad_w2 = h_out.T @ error_output
    grad_b2 = np.sum(error_output, axis=0, keepdims=True)

    error_hidden = error_output @ w2.T
    delta_hidden = error_hidden * relu_grad(h_lin)

    grad_w1 = inp.T @ delta_hidden
    grad_b1 = np.sum(delta_hidden, axis=0, keepdims=True)

    return {
        "grad_w1": grad_w1,
        "grad_b1": grad_b1,
        "grad_w2": grad_w2,
        "grad_b2": grad_b2
    }

# ---------------------- TRAINING FUNCTION ----------------------
def train_network(inp, true_val, hidden_units=4, lr=0.1,
                  epochs=5000, seed=10, show_every=500):

    np.random.seed(seed)

    w1 = np.random.randn(inp.shape[1], hidden_units) * 0.5
    b1 = np.zeros((1, hidden_units))

    w2 = np.random.randn(hidden_units, 1) * 0.5
    b2 = np.zeros((1, 1))

    loss_history = []

    for step in range(1, epochs + 1):

        # Forward
        pred, memory = forward_step(inp, w1, b1, w2, b2)
        loss_val = bce_loss(true_val, pred)
        loss_history.append(loss_val)

        # Backward
        gradients = backward_step(inp, true_val, memory, w2)

        # Parameter Update
        w1 -= lr * gradients["grad_w1"]
        b1 -= lr * gradients["grad_b1"]
        w2 -= lr * gradients["grad_w2"]
        b2 -= lr * gradients["grad_b2"]

        if step % show_every == 0 or step == 1:
            print(f"Step {step:4d} | Loss = {loss_val:.4f}")

    parameters = {"w1": w1, "b1": b1, "w2": w2, "b2": b2}
    return parameters, loss_history

# ---------------------- TRAIN MODEL ----------------------
params, losses = train_network(features, targets)

# ---------------------- LOSS CURVE ----------------------
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("Model Training Loss")
plt.grid(True)
plt.show()

# ---------------------- FINAL PREDICTIONS ----------------------
w1 = params["w1"]
b1 = params["b1"]
w2 = params["w2"]
b2 = params["b2"]

final_pred, _ = forward_step(features, w1, b1, w2, b2)
labels = (final_pred >= 0.5).astype(int)

print("\nPredicted Probabilities:\n", final_pred)
print("\nPredicted Labels:\n", labels)
print("\nActual Labels:\n", targets.astype(int))